# Opérateurs

In [1]:
import os
import re
import sqlite3

DUMP = "../../../data/northwind.sql"
DB = "northwind.sqlite"

with open(DUMP) as f:
    sql = f.read()

sql = re.sub(r"^SET .*?;\s*$", "", sql, flags=re.M)
sql = re.sub(r"ALTER TABLE ONLY[^;]*;", "", sql, flags=re.S)
sql = sql.replace("bytea", "BLOB")
sql = sql.replace(r"'\x'", "NULL")

if os.path.exists(DB):
    os.remove(DB)
conn = sqlite3.connect(DB)
conn.executescript(sql)
conn.commit()

cur = conn.cursor()
for table in ["customers", "orders", "order_details", "products", "employees"]:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"{table}: {cur.fetchone()[0]}")

customers: 91
orders: 830
order_details: 2155
products: 77
employees: 9


## Exercice 1 
- En utilisant la base de données Northwind, identifier les commandes (orders) pour lesquelles la quantité totale (voir les valeurs de la variable quantity dans order_details) est supérieure à 100
- On renverra juste deux colonnes dans les résultats :
    - order_id
    - une colonne qui vaut 'Large Order' (=>100) ou 'Small Order' (<100)

In [2]:
# 1. On ouvre la connexion
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

# 2. Écriture et exécution de la requête SQL

query = """
    SELECT order_id,
        CASE 
            WHEN SUM(quantity) >= 100 THEN 'Large Order'
            ELSE 'Small Order' 
        END AS order_size
    FROM order_details
    GROUP BY order_id
    LIMIT 10;
"""

cur.execute(query)
results = cur.fetchall()

# affichage 
for row in results:
    print(row)

# 4. Fermeture de la connexion
conn.close()

(10248, 'Small Order')
(10249, 'Small Order')
(10250, 'Small Order')
(10251, 'Small Order')
(10252, 'Large Order')
(10253, 'Large Order')
(10254, 'Small Order')
(10255, 'Large Order')
(10256, 'Small Order')
(10257, 'Small Order')


## Exercice 2

- Sélectionner toutes les commandes (orders), en affichant leur région de livraison, et en remplaçant les régions manquantes par 'Not Specified'. Afficher également la quantité totale commandée pour chaque commande (order_details)



In [3]:
# 1. On ouvre la connexion
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

# 2. Requête SQL avec COALESCE, JOIN et GROUP BY
query = """
SELECT 
    o.order_id,
    COALESCE(o.ship_region, 'Not Specified') AS region_livraison,
    SUM(d.quantity) AS quantite_totale
FROM orders o
INNER JOIN order_details d ON o.order_id = d.order_id
GROUP BY o.order_id
LIMIT 10;
"""

cur.execute(query)
results = cur.fetchall()

# affichage 
for row in results:
    print(row)

# 4. Fermeture de la connexion
conn.close()

(10248, 'Not Specified', 27)
(10249, 'Not Specified', 49)
(10250, 'RJ', 60)
(10251, 'Not Specified', 41)
(10252, 'Not Specified', 105)
(10253, 'RJ', 102)
(10254, 'Not Specified', 57)
(10255, 'Not Specified', 110)
(10256, 'SP', 27)
(10257, 'TÃ¡chira', 46)


# Doublons

## Exercice 3

- Créer une nouvelle table avec ce code :
- 
```sql
CREATE TABLE people(
    id SERIAL PRIMARY KEY,
    firstname VARCHAR(50) NOT NULL
);

-- 1 fois
INSERT INTO people(firstname) values('Maxime');

-- tous les autres en doublons
INSERT INTO people(firstname) values('Max');
INSERT INTO people(firstname) values('Marie');
INSERT INTO people(firstname) values('Marion');

INSERT INTO people(firstname) values('Max');
INSERT INTO people(firstname) values('Marion');
INSERT INTO people(firstname) values('Marie');

INSERT INTO people(firstname) values('Marie');


-- Vérifier : 
SELECT
    firstname, 
    COUNT(firstname)
FROM
    people
GROUP BY firstname;
```

- Écrire une requête utilisant GROUP BY et HAVING pour identifier les doublons

- Supprimer les doublons en utilisant ces différentes méthodes :
    - Une table intermédiaire :
        - On écrit les valeurs uniques dans une nouvelle table avec : `CREATE TABLE people_temp(LIKE people);`
        - On supprime l'ancienne (people)
        - On renomme la nouvelle
    - Une clause WHERE


In [11]:
import psycopg2

import os
from dotenv import load_dotenv
# Ajouter aux variables environnementales celles contenues dans le .env
load_dotenv("../../../.env")

USER = "postgres" #tjrs "postgres"
PASSWORD = ""

try:
    # Connexion à PostgreSQL
    conn = psycopg2.connect(user=USER, password="Mkilo1990", host="localhost", port=5432)
    conn.autocommit = True
    cur = conn.cursor()

    # Pour pouvoir ré exécuter le code
    cur.execute("DROP DATABASE IF EXISTS northwind;")

    # Création d'une nouvelle base de données
    cur.execute("CREATE DATABASE northwind;")
    print("Database created successfully.")

    # Connexion à la nouvelle base de données
    conn.close()


except psycopg2.Error as e:
    print(f"Error connecting to PostgreSQL database: {e}")

conn = psycopg2.connect(
    dbname="northwind",
    user=USER,
    password="Mkilo1990",
    host="localhost",
    port=5432
)

cursor = conn.cursor()

DUMP = "../../../data/northwind.sql"

with open(DUMP, "r", encoding="utf-8") as f:
    sql = f.read()

cursor.execute(sql)

conn.commit()

cursor.close()
conn.close()

print("SQL file imported successfully.")

Database created successfully.
SQL file imported successfully.
